In [ ]:
%config InlineBackend.figure_format = "retina"

import pandas as pd
import importlib
import matplotlib.pyplot as plt
import gtfsutil
import numpy as np
from functools import partial
import re
import keyring

In [ ]:
gtfsutil = importlib.reload(gtfsutil)

In [ ]:
orig = gtfsutil.read_gtfs("mvta.zip")
mod = gtfsutil.read_gtfs("transit-scenario.zip")

In [ ]:
def marey_plot(route_id, dir_id, date):
    all_stop_list = []
    
    stop_trip_index = dict()
    
    # find all the stops, more or less in order - only need to do with original feed, as new feed
    # is just duplicates
    orig_trips = orig.trips[(orig.trips.route_id == route_id) & (orig.trips.direction_id == dir_id) &
                           (orig.trips.service_id.apply(partial(gtfsutil.service_running, orig, date)))].index
    
    for tid in orig_trips:
        last_stop_location = -1
        for stop in orig.stop_times.loc[tid].stop_id:
            if stop in all_stop_list:
                last_stop_location = all_stop_list.index(stop)
            else:
                all_stop_list.insert(last_stop_location + 1, stop)
                last_stop_location += 1
                
    
    # now, plot each trip
    for tid in orig_trips:
        st = orig.stop_times.loc[tid]
        xs = st.departure_time.apply(gtfsutil.gtfs_time_to_seconds) / 3600
        ys = st.stop_id.apply(all_stop_list.index)
        plt.plot(xs, ys, lw=3, color="blue")
        

    mod_trips = mod.trips[(mod.trips.route_id == route_id) & (mod.trips.direction_id == dir_id) &
                           (mod.trips.service_id.apply(partial(gtfsutil.service_running, mod, date)))].index
    
    print(f"{len(orig_trips)} original trips, {len(mod_trips)} mod trips")
    
    for tid in mod_trips:
        st = mod.stop_times.loc[tid]
        xs = st.departure_time.apply(gtfsutil.gtfs_time_to_seconds) / 3600
        ys = st.stop_id.apply(all_stop_list.index)
        plt.plot(xs, ys, lw=1, color="red")
        
    
    # label some stops
    stops = orig.stops.set_index("stop_id")
    stop_names = [stops.stop_name.at[sid] for sid in all_stop_list]
    step = max(len(stop_names) // 10, 1)
    name_idxs = np.r_[:len(all_stop_list):step]
    plt.yticks(name_idxs, [stop_names[i] for i in name_idxs])
    plt.xlabel("Hour of day")
    
    # figure out route name and direction for title
    rid = orig.trips.at[orig_trips[0], "route_id"]
    routes = orig.routes.set_index("route_id")
    route_short_name = routes.at[rid, "route_short_name"]
    route_long_name = routes.at[rid, "route_long_name"]
    headsign = orig.trips.at[orig_trips[0], "trip_headsign"]
    if m := re.search("[a-zA-Z]+bound", headsign):
        dirname = m[0]
    else:
        dirname = ""
        
    day_of_week = gtfsutil.gtfs_date_to_date(date).strftime("%A")
        
    plt.title(f"Route {route_short_name} {route_long_name} {dirname}, {day_of_week}")
    
    
                               
    
                    

In [ ]:
plt.figure(figsize=(16, 8))
marey_plot("12-112", 0, 20191023)

In [ ]:
plt.figure(figsize=(16, 8))
marey_plot("600-112", 1, 20191023)

In [ ]:
orig.trips[orig.trips.route_id == "600-112"]

In [ ]:
with pd.option_context("display.max_rows", 10000):
    display(mod.stop_times.loc[mod.trips[(mod.trips.route_id == "12-112") & (mod.trips.service_id == "thursday")].index])

In [ ]:
orig.stop_times.loc[orig.trips[(orig.trips.route_id == "600-112")].index]